# Beginner tour 4: Choosing molecules, and combining backbones

After training (notebook 3), the model gives every undocked molecule a guess `mu` and an
uncertainty `sigma`. Two questions remain, and they're what most of the project's research
is about:

1. **Which molecules should we dock next?** Should we trust the best guesses, or also try
   molecules the model is unsure about?
2. **We have three backbones, each seeing molecules differently. Can we combine them?**

You'll answer both with real experiments on the real data. Some cells run several full
campaigns and take a minute or two, about 6 minutes for the whole notebook.

In [ ]:
import sys, json, io, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tour import *

import run_experiment as rx
import surrogates
from molpal.acquirer.metrics import greedy, ucb
register_mini(rx)

scores, embeddings = load_mini()
y = -scores["score"].to_numpy(np.float32)            # flipped: higher = better (as in notebook 3)
best100 = set(np.argsort(-y)[:100].tolist())         # positions of the true best 100
print("ready")

---
## 1. Explore or exploit?

Imagine choosing a restaurant. You can go to your favourite (**exploit** what you know), or
try a new place that *might* be better (**explore**). Always exploiting means you never find
anything better. Always exploring means you rarely enjoy a good meal.

Choosing molecules is the same. The project's two rules, called **acquisition functions**,
come from `molpal/acquirer/metrics.py`:

* **greedy**: rank by `mu`, the guess. Pure exploitation.
* **UCB** (upper confidence bound): rank by `mu + 2 × sigma`, "how good it *could* be".
  Molecules the model is unsure about get a bonus. That's exploration.

🐍 **Python corner: importing functions.** `from molpal.acquirer.metrics import greedy, ucb`
brings in two functions written by the project. They're ordinary functions: you give them
arrays and get arrays back. (`ucb` expects the *variance*, `sigma ** 2`.)

In [ ]:
names = ["A", "B", "C", "D"]
mu    = np.array([10.0, 9.5, 9.0, 6.0])
sigma = np.array([ 0.1, 0.2, 1.5, 0.3])
print("molecule:  ", names)
print("greedy:    ", greedy(mu))
print("ucb:       ", ucb(mu, sigma ** 2).round(1))
print("greedy picks", names[np.argmax(greedy(mu))], "| ucb picks", names[np.argmax(ucb(mu, sigma ** 2))])

Greedy picks A, the best guess. UCB picks C: its guess is a bit lower, but the model is very
unsure about it, so it *might* be excellent.

### On real data: one round

Train the Uni-Mol2 surrogate on 210 random molecules (like round 0), then let each rule pick
a batch of 210. How many of the true best 100 does each batch contain?

In [ ]:
rng = np.random.default_rng(0)
order = rng.permutation(len(y))
labeled, candidates = order[:210], order[210:]

torch.manual_seed(0)
X = embeddings["unimol2"]
model = surrogates.SingleBackboneMVESurrogate(in_dim=X.shape[1])
model.fit(X[labeled], y[labeled], epochs=50)
mu, sigma = model.predict(X[candidates])

pick_greedy = set(candidates[np.argsort(-greedy(mu))[:210]].tolist())
pick_ucb    = set(candidates[np.argsort(-ucb(mu, sigma ** 2))[:210]].tolist())
print("best-100 molecules in the greedy batch:", len(pick_greedy & best100))
print("best-100 molecules in the UCB batch:   ", len(pick_ucb & best100))
print("the two batches share", len(pick_greedy & pick_ucb), "of 210 molecules")

🐍 **Python corner: sets.** `a & b` is the **intersection** of two sets: the items in both.
So `pick_greedy & best100` is "the molecules that are both in the greedy batch and among the
true best".

One round only tells part of the story. Exploration might pay off **later**: what UCB learns
from its unusual picks could make the *next* rounds better. The only way to know is to run
whole campaigns.

---
## 2. Whole campaigns: greedy vs UCB

The same helper as in notebook 2, now with a choice of surrogate and backbones. Each call
runs a complete 5-round campaign with the project's real `main()`.

In [ ]:
def run(surrogate="single", backbones=("unimol2",), seed=42, acq="greedy"):
    out = new_run_dir()
    which = ["--backbone", backbones[0]] if surrogate == "single" else ["--backbones", *backbones]
    sys.argv = ["run_experiment.py", "--dataset", "MINI", "--mode", "mve",
                "--surrogate", surrogate, *which, "--acq", acq,
                "--init-size", "210", "--batch-size", "210", "--n-rounds", "5",
                "--topk", "100", "--seed", str(seed), "--run-dir", str(out)]
    with contextlib.redirect_stdout(io.StringIO()):
        rx.main()
    return json.loads((out / "history.json").read_text())[-1]["topk_recall"]

SEEDS = [42, 43, 44]
results = {}                                          # we'll collect every strategy's results here
results["Uni-Mol2, greedy"] = [run(seed=s) for s in SEEDS]
results["Uni-Mol2, UCB"]    = [run(seed=s, acq="ucb") for s in SEEDS]
for name, r in results.items():
    print(f"{name:18s} final recall {np.mean(r):.3f}   (seeds: {r})")

Exploring didn't help here. UCB ends up about the same as greedy, or slightly worse. This is
one of the project's main findings, on the full 2.1-million-molecule pool too: after 145
carefully planned runs testing every way of tuning exploration (fixed amounts, schedules,
even learned controllers), **none beat plain greedy**
(`al-molecular/docs/adaptive_acq_results.md`, §3). Notebook 6 shows those controllers.

Why? With a decent model, the best guesses are already very good, and the batches are large
enough to contain some "surprises" anyway. Exploration spends budget on molecules that are
usually just average.

---
## 3. Combining backbones, idea 1: let them vote

Each backbone sees molecules differently: text, 3D shape, graph. Maybe together they see
more. The project's first combination, `EnsembleFusionSurrogate` (`--surrogate ensemble`),
trains **one model per backbone** and lets them **vote with ranks**:

1. each backbone's model ranks all molecules (rank 1 = its best guess)
2. a molecule's combined score is its **average rank** across backbones
3. the **disagreement** between backbones becomes the uncertainty `sigma`

Here's the project's real voting function, `combine`, on 5 made-up molecules:

In [ ]:
votes = np.array([      # each backbone's guess for 5 molecules (higher = better)
    # molformer unimol2 mhgged
    [9.0,       8.8,    9.1],     # m0: all three love it
    [5.0,       5.2,    4.9],     # m1: all three dislike it
    [2.0,       9.5,    9.0],     # m2: molformer dislikes it, the other two love it
    [7.0,       7.1,    6.9],     # m3: everyone says "OK"
    [9.9,       2.0,    2.5],     # m4: only molformer loves it
], dtype=np.float32)
fusion = surrogates.EnsembleFusionSurrogate(dims={"molformer": 1, "unimol2": 1, "mhgged": 1})
combined, disagreement = fusion.combine(votes)
for i in range(5):
    print(f"m{i}: combined score {combined[i]:6.2f}   disagreement {disagreement[i]:.3f}")

Note m2: two good backbones love it, but MoLFormer's single vote drags it down. In an equal
vote, **a weak backbone counts as much as a strong one.** And notebook 3 showed MoLFormer is
much weaker on this task. Let's see what that does to real campaigns:

In [ ]:
ALL = ("molformer", "unimol2", "mhgged")
results["vote, all 3"]           = [run("ensemble", ALL, s) for s in SEEDS]
results["vote, without MoLFormer"] = [run("ensemble", ("unimol2", "mhgged"), s) for s in SEEDS]
for name in ["vote, all 3", "vote, without MoLFormer"]:
    print(f"{name:24s} final recall {np.mean(results[name]):.3f}   (seeds: {results[name]})")

A huge difference. Voting with all three is **much worse** than Uni-Mol2 alone, but voting
without MoLFormer is as good or better. One bad voter spoils the result. In real research
you don't always know in advance which backbone is weak, so we want a combination that
**figures it out by itself**.

---
## 4. Combining backbones, idea 2: learn how much to trust each

`LearnedFusionSurrogate` (`--surrogate learned`) still trains one model per backbone, but
then **learns a weight for each** from the data: combined guess = w₁ × MoLFormer's guess +
w₂ × Uni-Mol2's + w₃ × MHG-GED's. It learns the weights with a linear model (RidgeCV), using
molecules held back from training so the weights are honest.

Let's train one on 420 molecules and look at the weights it chose:

In [ ]:
dims = {k: embeddings[k].shape[1] for k in ALL}
X_all = np.concatenate([embeddings[k] for k in ALL], axis=1)     # the three tables side by side
train = order[:420]

torch.manual_seed(0)
learned = surrogates.LearnedFusionSurrogate(dims=dims)
with contextlib.redirect_stdout(io.StringIO()):
    learned.fit(X_all[train], y[train])
for name, w in zip(ALL, learned._meta.coef_):
    print(f"weight for {name:10s} {w:6.3f}")

The model worked it out by itself: **MoLFormer gets a weight of about 0**. It's effectively
ignored, while Uni-Mol2 and MHG-GED share the work.

🐍 **Python corner: `np.concatenate(..., axis=1)`** glues tables side by side (`axis=1` =
columns). The fusion surrogates take one wide table and split it back into backbones
internally, which is why they need `dims`, the width of each piece.

In [ ]:
results["learned weights, all 3"] = [run("learned", ALL, s) for s in SEEDS]
print(f"learned weights, all 3   final recall {np.mean(results['learned weights, all 3']):.3f}   "
      f"(seeds: {results['learned weights, all 3']})")

---
## 5. Combining backbones, idea 3: one big model sees everything (LT-All)

The project's current default, `LTAllSurrogate` (`--surrogate ltall`), takes a different
approach. Instead of one model per backbone, it feeds **all** embeddings, glued together
(768 + 768 + 1,024 = 2,560 numbers per molecule), into **one** larger network. It also
trains **5 copies** of that network from different random starts, and their disagreement
becomes `sigma`.

In [ ]:
results["LT-All, all 3"] = [run("ltall", ALL, s) for s in SEEDS]
print(f"LT-All, all 3   final recall {np.mean(results['LT-All, all 3']):.3f}   (seeds: {results['LT-All, all 3']})")

LT-All also copes well with the weak backbone. It has a built-in learnable weight per
backbone, but here's an honest surprise: look at what those weights end up as.

In [ ]:
torch.manual_seed(0)
ltall = surrogates.LTAllSurrogate(dims=dims)
with contextlib.redirect_stdout(io.StringIO()):
    ltall.fit(X_all[train], y[train])
net = ltall._models[0]                                   # the first of the 5 networks
w = torch.softmax(net.fusion.logits / net.fusion._tau, dim=0)
print({name: round(float(v), 3) for name, v in zip(ALL, w)})

They barely moved from ⅓ each. So whatever lets LT-All cope with MoLFormer, it isn't those
weights. The most likely explanation (not measured here) is its first big layer: with
2,560 inputs → 1,024 outputs it has a separate weight for every input column, so it can learn
to rely on the useful columns and mostly ignore the rest. This is a good habit: when
a design says "this part does X", **check whether it actually does**.

---
## 6. All strategies side by side

In [ ]:
table = pd.DataFrame({name: {"mean recall": np.mean(r), "spread": np.std(r)} for name, r in results.items()}).T
table = table.sort_values("mean recall")
display(table.round(3))

plt.figure(figsize=(7, 3.5))
plt.barh(table.index, table["mean recall"], xerr=table["spread"], color="tab:blue")
plt.axvline(1260 / len(y), color="gray", linestyle="--", label="random docking")
plt.xlabel("fraction of the best 100 found (3 seeds)"); plt.xlim(0, 1); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()

🐍 **Python corner: `display(...)`** shows a table nicely in a notebook, the same as leaving
it as the last line of a cell.

**What this tells us, all on real data:**

* Every strategy beats random docking by a mile.
* **Exploring (UCB) didn't help.** Greedy is as good or better, as the project found at full scale.
* **Combining backbones can hurt** if done naively (equal votes), because one weak backbone
  drags the others down.
* **Learned combinations** (learned weights, LT-All) find out by themselves which backbones
  to trust, and match or beat the best single backbone. That's why LT-All became the
  project's default.

Three seeds and a small pool, so differences of 0.01–0.02 between the top strategies are
within noise. The big gaps are real.

The project's own history follows the same arc: rank voting (method [3]) → learned weights →
LT-All → (notebook 6) smarter acquisition, which didn't pan out → (the actual win) better
training targets and better embeddings.

---
## Summary

| Idea | In the code | Lesson on real data |
|---|---|---|
| greedy: trust the guess | `metrics.greedy`, `--acq greedy` | as good as anything tried |
| UCB: bonus for uncertainty | `metrics.ucb`, `--acq ucb` | no better than greedy |
| equal rank vote | `EnsembleFusionSurrogate`, `--surrogate ensemble` | hurt by a weak backbone |
| learned weights | `LearnedFusionSurrogate`, `--surrogate learned` | learns to ignore the weak backbone |
| one model on everything | `LTAllSurrogate`, `--surrogate ltall` | robust, and the project's default |

🐍 **Python you met:** importing specific functions, set intersection `&`, a dict of lists
for results, `np.concatenate`, reading a model's learned weights, `pd.DataFrame(...).T`
(`.T` flips rows and columns), `display`.

## Try it yourself
Answers, run on the same real data: [solutions/4_solutions.ipynb](solutions/4_solutions.ipynb). Try first!

1. Run `run("ltall", ("unimol2", "mhgged"), 42)`. Does removing MoLFormer still matter for LT-All?
2. Run `run("ensemble", ALL, 42, acq="ucb")`. For rank voting, `sigma` is backbone disagreement: does exploring it help?
3. Add a fourth strategy to the table: `run("single", ("mhgged",), s)` for each seed.

**Next:** `5` is `../00_how_al_molecular_works/classes_and_call_flow.ipynb`, how all these
classes are wired together in the code. Then `6_smarter_strategies.ipynb`.